# 3.0 Warning pipeline with linear models and baselines, 30 and 60 minutes
**Purpose.** Run the whole pipeline end to end with simple models: labels on 2022 and 2023, features, the three warning approaches with linear models, and the two baselines.

**Method.** Untuned linear models: ordinary least squares for the flow and occupancy forecasts, and L2 logistic regression with C = 1 for direct classification. Baselines: proximity (relative occupancy at *t*) and calendar (the 2022 share of positive origins per district × weekday/weekend × slot of day). Every threshold, flow reference, calendar share and model parameter is fixed on 2022. The warning thresholds of all scores are calibrated in notebook 3.4.

**Outputs.** The design manifest, the 2022 references and the calendar shares go to `models/`, and the 2022 and 2023 scores to `data/interim/scores-linear.parquet`. 2024 is not read.

## 1. Setup

In [1]:
import polars as pl

from madrid_congestion.config import HORIZONS, INTERIM_DATA_DIR, MODELS_DIR, PANEL_YEAR, PROCESSED_DATA_DIR, VALIDATION_YEAR
from madrid_congestion.evaluation import forecast_mae, warning_table
from madrid_congestion.features import feature_names, origin_rows
from madrid_congestion.frozen import write_design
from madrid_congestion.labels import (
    SLOT_OF_DAY,
    WEEKEND,
    add_labels,
    baseline_scores,
    calendar_baseline,
    references,
    studied_districts,
    year_grid,
)
from madrid_congestion.models import LINEAR_MODELS, fit_model, fit_target, score_model
pl.Config.set_tbl_rows(-1)
pl.Config.set_tbl_cols(-1)
series = {year: pl.read_parquet(PROCESSED_DATA_DIR / f"district-series-{year}.parquet") for year in (PANEL_YEAR, VALIDATION_YEAR)}

## 2. Labels on both years, with the 2022 rules
The studied districts, thresholds and flow references come from 2022 and are applied unchanged to 2023. The package must reproduce paper Table 3.3.

In [2]:
districts = studied_districts(series[PANEL_YEAR])  # the 19 districts with at least half of their 2022 slots valid
refs = references(year_grid(series[PANEL_YEAR], districts, PANEL_YEAR))  # 2022 thresholds and flow references
labels = {year: add_labels(year_grid(series[year], districts, year), refs) for year in series}

# The package must give the same 2022 numbers as paper Table 3.3 (notebook 2.0)
check = labels[PANEL_YEAR].select(
    onsets=pl.col("onset").sum(),
    origins_30=pl.col("usable_30").sum(),
    positive_30=(pl.col("usable_30") & pl.col("target_30")).sum(),
    origins_60=pl.col("usable_60").sum(),
    positive_60=(pl.col("usable_60") & pl.col("target_60")).sum(),
).row(0)
assert check == (5_466, 475_434, 8_080, 453_544, 15_510), check

pl.DataFrame([
    {"year": year, "minutes": minutes, "onsets": df["onset"].sum(), "usable_origins": df[f"usable_{minutes}"].sum(),
     "positive": (df[f"usable_{minutes}"] & df[f"target_{minutes}"]).sum()}
    for year, df in labels.items() for minutes in HORIZONS
]).with_columns(base_rate=(pl.col("positive") / pl.col("usable_origins")).round(4))

year,minutes,onsets,usable_origins,positive,base_rate
i64,i64,i64,i64,i64,f64
2022,30,5466,475434,8080,0.017
2022,60,5466,453544,15510,0.0342
2023,30,5755,482391,8380,0.0174
2023,60,5755,461884,16275,0.0352


## 3. Features and rows
The 34 inputs of paper Table 3.4: relative occupancy and flow at *t*, *t*−1, *t*−2 and *t*−3, slot of day as sine and cosine, day of week and district. One row is one usable calm origin at the horizon.

In [3]:
FEATURES = feature_names(districts)
rows = {(year, minutes): origin_rows(df, districts, minutes) for year, df in labels.items() for minutes in HORIZONS}
for df in rows.values():
    assert df.select(FEATURES).null_count().sum_horizontal().item() == 0  # usable origins have a full valid hour
print(len(FEATURES), "features |", {key: df.height for key, df in rows.items()}, "rows")

34 features | {(2022, 30): 475434, (2022, 60): 453544, (2023, 30): 482391, (2023, 60): 461884} rows


## 4. Models, fitted on 2022
One model per approach and horizon, trained on the 2022 rows of all districts together (`fit_model`). Inputs are standardised inside the pipeline, so the scaling is learned from 2022 only. The two forecasts predict the largest relative value in *t*+1 … *t*+*H*, and the classifier predicts the target. Nothing is tuned.

In [4]:
models = {}
for minutes in HORIZONS:
    for name in LINEAR_MODELS:  # occupancy and flow: OLS forecasts; direct: L2 logistic classifier
        models[minutes, name] = fit_model("linear", name, minutes, rows[PANEL_YEAR, minutes], districts)
    print(f"{minutes} min: logistic regression iterations {models[minutes, 'direct'][-1].n_iter_[0]}")

30 min: logistic regression iterations 23
60 min: logistic regression iterations 26


## 5. Warning scores
Every approach and baseline gives one score per origin, and a higher score is a stronger warning. The proximity baseline is the relative occupancy at *t*. The calendar baseline is the 2022 share of positive origins for the origin's district, day type and slot of day at the same horizon.

In [5]:
calendar = calendar_baseline(labels[PANEL_YEAR], districts)

scores = {}
for (year, minutes), df in rows.items():
    scores[year, minutes] = pl.concat(
        [
            df.select("distrito", "slot_start", target=pl.col(f"target_{minutes}")),
            baseline_scores(df, calendar, minutes),  # proximity and calendar, in row order
            pl.DataFrame({f"linear_{name}": score_model(models[minutes, name], name, df, districts) for name in LINEAR_MODELS}),
        ],
        how="horizontal",
    )

# The fallback: a cell without any usable 2022 origin carries its district x day-type share of 2022, nothing else
CALENDAR_KEYS = ["distrito", "weekend", "slot_of_day"]
empty = calendar.filter(pl.col("origins") == 0)
assert (empty["calendar_rate"] == empty["day_type_rate"]).all()
for (year, minutes), df in scores.items():
    fallback = df.with_columns(weekend=WEEKEND, slot_of_day=SLOT_OF_DAY).join(
        empty.filter(pl.col("horizon") == minutes).select(CALENDAR_KEYS), on=CALENDAR_KEYS, how="semi"
    )
    print(f"{year}, {minutes} min: {fallback.height} origins scored with the fallback share, {fallback['target'].sum()} of them positive")
print(empty.group_by("horizon").agg(empty_cells=pl.len(), fallback_share_zero=(pl.col("calendar_rate") == 0).sum()).sort("horizon"))

2022, 30 min: 0 origins scored with the fallback share, 0 of them positive
2022, 60 min: 0 origins scored with the fallback share, 0 of them positive
2023, 30 min: 4 origins scored with the fallback share, 0 of them positive
2023, 60 min: 4 origins scored with the fallback share, 0 of them positive
shape: (2, 3)
┌─────────┬─────────────┬─────────────────────┐
│ horizon ┆ empty_cells ┆ fallback_share_zero │
│ ---     ┆ ---         ┆ ---                 │
│ i32     ┆ u32         ┆ u32                 │
╞═════════╪═════════════╪═════════════════════╡
│ 30      ┆ 109         ┆ 39                  │
│ 60      ┆ 129         ┆ 45                  │
└─────────┴─────────────┴─────────────────────┘


## 6. AP on 2022 and 2023
AP measures how well the warning scores rank positive origins first, and it needs no threshold. A random ranking has an expected AP close to the base rate. Every count is a count of usable calm origins, not of onsets. 2022 is the training year and is shown for comparison only.

In [6]:
SCORES = [f"linear_{name}" for name in LINEAR_MODELS] + ["proximity", "calendar"]
ap = []
for (year, minutes), df in scores.items():
    table = warning_table(df["target"], {name: df[name] for name in SCORES}).select("score", "ap")
    ap.extend({"minutes": minutes, "year": year, **row} for row in table.iter_rows(named=True))
print({key: round(float(df["target"].to_numpy().mean()), 4) for key, df in scores.items()}, "base rates")
(
    pl.DataFrame(ap)
    .pivot("year", index=["minutes", "score"], values="ap")
    .rename({str(PANEL_YEAR): "ap_2022", str(VALIDATION_YEAR): "ap_2023"})
    .with_columns(pl.col("ap_2022", "ap_2023").round(4))
)

{(2022, 30): 0.017, (2022, 60): 0.0342, (2023, 30): 0.0174, (2023, 60): 0.0352} base rates


minutes,score,ap_2022,ap_2023
i64,str,f64,f64
30,"""linear_occupancy""",0.4193,0.4149
30,"""linear_flow""",0.3604,0.3576
30,"""linear_direct""",0.5228,0.5172
30,"""proximity""",0.2471,0.2483
30,"""calendar""",0.3758,0.3521
60,"""linear_occupancy""",0.4593,0.4549
60,"""linear_flow""",0.3937,0.3851
60,"""linear_direct""",0.5544,0.5503
60,"""proximity""",0.2678,0.2792


**Forecast accuracy.** MAE of each forecast against the realised largest value in *t*+1 … *t*+*H*, in relative units and in original units (each value multiplied by its district's 2022 reference: percentage points of occupancy and vehicles per hour). The naive forecast, the value at *t*, is the reference a forecast has to beat.

In [7]:
mae = []
for (year, minutes), df in rows.items():
    for variable, reference in (("occupancy", "threshold"), ("flow", "flow_reference")):
        mae.append({
            "minutes": minutes, "year": year, "forecast": variable,
            **forecast_mae(scores[year, minutes][f"linear_{variable}"], df[fit_target(variable, minutes)], df[f"{variable}_lag0"], df[reference]),
        })
pl.DataFrame(mae).sort("minutes", "forecast", "year").with_columns(pl.exclude("minutes", "year", "forecast").round(4))

minutes,year,forecast,mae_relative,naive_mae_relative,mae_original_units,naive_mae_original_units,mae_reduction
i64,i64,str,f64,f64,f64,f64,f64
30,2022,"""flow""",0.0322,0.0395,20.2398,24.6971,0.1847
30,2023,"""flow""",0.0312,0.0386,19.552,24.1034,0.1915
30,2022,"""occupancy""",0.0407,0.0508,0.5049,0.6265,0.1993
30,2023,"""occupancy""",0.041,0.0511,0.5119,0.635,0.1987
60,2022,"""flow""",0.0466,0.0587,29.1008,36.295,0.2067
60,2023,"""flow""",0.0455,0.0579,28.4099,35.8071,0.2138
60,2022,"""occupancy""",0.0563,0.0751,0.6924,0.9202,0.2509
60,2023,"""occupancy""",0.0566,0.0758,0.7005,0.9343,0.2537


## 7. Frozen outputs
The design manifest (`models/design.json`), which every later notebook and the 2024 run check against the code, the 2022 references and calendar shares, which the 2024 run loads, and the scores of both years for notebook 3.4.

In [8]:
write_design(districts)
refs.write_parquet(MODELS_DIR / "references-2022.parquet")
calendar.write_parquet(MODELS_DIR / "calendar-2022.parquet")
INTERIM_DATA_DIR.mkdir(exist_ok=True)
pl.concat([df.select(pl.lit(year).alias("year"), pl.lit(minutes).alias("minutes"), pl.all()) for (year, minutes), df in scores.items()]).write_parquet(
    INTERIM_DATA_DIR / "scores-linear.parquet"
)
refs

distrito,valid_slots,threshold,flow_reference
i64,u32,f64,f64
1,30244,18.542553,687.029703
2,29277,12.243902,875.6625
3,33055,12.266667,876.684211
4,33559,13.64,977.047244
5,32515,13.80814,845.034682
6,33207,9.537879,731.080882
7,33401,12.131313,972.342593
8,31163,9.814035,455.73064
9,29841,12.587302,617.76378


**Result (2023 validation, not test results).** The 2023 AP of the linear models and the baselines, with base rates of 1.74% (30 min) and 3.52% (60 min):

| Score | 30 min | 60 min |
|---|---|---|
| Linear direct (logistic) | 0.517 | 0.550 |
| Linear occupancy forecast (OLS) | 0.415 | 0.455 |
| Linear flow forecast (OLS) | 0.358 | 0.385 |
| Calendar baseline | 0.352 | 0.450 |
| Proximity baseline | 0.248 | 0.279 |

Every score lies far above the base rate, and the 2022 and 2023 AP are close, so the linear models degrade little between years. The linear forecasts reduce the naive MAE by about 20% at 30 minutes and 21–25% at 60 minutes (at 60 minutes: 0.70 against 0.93 percentage points of occupancy, and 28.4 against 35.8 vehicles per hour). The calendar fallback scores 4 origins of 2023 at each horizon, none of them positive.

**Conclusion.** The linear models and baselines are fixed for both horizons. 2023 is the validation year, so this ranking is not the comparison of the study, which is made once on 2024 (notebook 4.0).